## WELCOME TO NUDIMAX

Richard Baker,¹ Lynn J. Bonomo,² Paola Guzmán² ³, Terry Gosliner²

¹Center for Comparative Genomics, California Academy of Sciences

²Department of Invertebrate Zoology & Geology ([Gosliner Slug Lab](https://sluglab.wordpress.com/))

³University of Puerto Rico at Cayey

(click to show/hide documentation)

#### OVERVIEW
This Python notebook is designed to facilitate data-wrangling for partitioned maximum-likelihood phylogenetic analysis projects that follow a typical workflow of the Gosliner Slug Lab at the California Academy of Sciences (hence, NUDIMAX = NUDIbranch MAXimum-likelihood*). This is all packaged as an interactive Python notebook for use in [Google Colab](https://colab.research.google.com) (and includes a basic user interface using their Forms feature, but relies on an Internet connection) or a [Jupyter notebook](https://jupyter.org/) (which requires interacting directly with the code, but can be installed and run locally). It is also portable to an HPC cluster, if one is available.

> **\*Note:** Since v0.16, NUDIMAX supports both maximum-likelihood and Bayesian inference.

#### FEATURES
This is designed to:
- Receive a CSV matrix of accession numbers and GenBank IDs as input (see example Table 1)
- Process that CSV and output a list of GenBank accession numbers for submission to [BatchEntrez](https://www.ncbi.nlm.nih.gov/sites/batchentrez).
- Process the GenBank download and produce separate FASTA files for each gene
- Remotely query the NCBI BLAST database to verify that GenBank accessions were assigned correctly*
- Align those sequences (using MAFFT), returning both FASTA and NEXUS alignment files for partitioned maximum-likelihood analysis (using IQ-TREE)
- Combine the alignment files into a partitioned NEXUS for use in Bayesian analysis (using MrBayes)
- Run both alignment packages
- Rename Newick branches (e.g., adding species names)

> **\*Note:** While this feature is included, a remote BLAST query is much slower than the BLAST web interface or using a local database.

#### METHODOLOGY
This is accomplished via a number of custom functions, which primarily use Biopython and Pandas. Importantly, these custom functions only wrangle and convert data; NUDIMAX does not perform any analysis or computation directly. Instead, it includes user-friendly wrappers that allow users to call peer-reviewed packages (including MAFFT, IQ-TREE, MrBayes, and optionally NCBI BLAST) without requiring command-line interaction, root access, or macOS/Windows/Linux compatibility issues.

#### DISCLAIMER
This notebook is provided "as-is" with no warranty. While this tool is designed to automate and streamline a standard workflow, it remains the user's responsibility to validate all data entering and exiting the program. Users are **strongly** encouraged to read the code and comments to ensure that this pipeline meets the specific needs of their project.

NUDIMAX is still in active development, and should be considered to be in an alpha state. While it has been tested with a number of inputs, it may contain bugs as yet unseen (and therefore, unsquashed).

#### CITATION
Although NUDIMAX was written to handle the Gosliner Lab's specific use case, it should be helpful for anyone using a similar workflow. If you use NUDIMAX in your research, please cite the project as shown below.

**Important:** Since NUDIMAX includes wrappers for other utilities (including [MAFFT](https://mafft.cbrc.jp/alignment/software/), [NCBI BLAST](https://blast.ncbi.nlm.nih.gov/doc/blast-help/references.html#references), [IQ-TREE](https://iqtree.github.io/), and [MrBayes](https://nbisweden.github.io/MrBayes/)), you must also cite the utilities you use.

**APA format:**

Baker, R., Bonomo, L., Guzmán, P., & Gosliner, T. (2026). NUDIMAX: An accessible data-wrangling pipeline for partitioned phylogenetic workflows (Version 0.15a). Retrieved from https://github.com/RichardMSBS/NUDIMAX


**BibTeX format:**

```
bibtex
@software{NUDIMAX},
  author = {Baker, Richard and Bonomo, Lynn and Guzmán, Paola and Gosliner, Terry},
  title = {NUDIMAX: An accessible data-wrangling pipeline and wrapper for partitioned phylogenetic workflows},
  version = {0.17a},
  year = {2026},
  url = {https://github.com/RichardMSBS/NUDIMAX}
}
```

## Run this chunk to setup NUDIMAX
note: In Google Colab, collapse this section for one-click setup

In [13]:
!pip install git+https://github.com/RichardMSBS/nudimax.git
import nudimax
nudimax.setup()

  Cloning https://github.com/RichardMSBS/nudimax.git to /tmp/pip-req-build-7ach_kes
  Running command git clone --filter=blob:none --quiet https://github.com/RichardMSBS/nudimax.git /tmp/pip-req-build-7ach_kes
  Resolved https://github.com/RichardMSBS/nudimax.git to commit a8279072b34662937cf6d13e3a707c05fefd6f05
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for nudimax: filename=nudimax-0.16.2-py3-none-any.whl size=29207 sha256=6f1cb037036da95c2c38341655d58e4d6a72ee8f302df15dc7fab0678cf990e4
  Stored in directory: /tmp/pip-ephem-wheel-cache-3n11kb42/wheels/81/09/1f/75e771fa8fc11cfba7d1a362ca0ea6e1f7fd577dd2e392e28a
Successfully built nudimax
NUDIMAX: installing dependencies...
  ✓ biopython installed.
  ? attempting to detect environment...
  ✓ environment detected: COLAB
  ✓ ncbi-blast-2.17.0+ installed.
  ✓ mafft-linux64 installed.
  ✓ mrbayes-3.2.7 installed.
  ✓ iqtree-3.1.3-Li

## User-facing commands below
(Click to expand individual functions)

---

**CAUTION: DON'T LOSE YOUR DATA!**

**Google Colab's file storage is cleared every time the runtime is disconnected.**

Files are not saved unless you specifically mount (and save to) your Google Drive.

When working in the Colab temporary storage, remember to regularly download all outputs and log files.

### Google Colab: Export to Google Drive

In [ ]:
# export_to_drive() takes up to three inputs
#   destination   (Optional) is the target file path on Google Drive.
#                 Defaults to NUDIMAX_backup_<timestamp>
#   source        is the file or folder to back up
#   copy_all      (Optional) will copy the entire NUDIMAX scratch folder.

In [ ]:
# @title export_to_drive() {single-column:true}
# @markdown Destination filepath on Google Drive (Default: NUDIMAX_backup_yymmdd-hhmmss)
destination = "" # @param {"type":"string", "placeholder":"Tenellia_backup"}
# @markdown Source
source = '' # @param {"type":"string", "placeholder":"Tenellia_backup"}
# @markdown Copy entire Google Colab scratch directory?
copy_all = False # @param {"type":"boolean"}
export_to_drive(destination = destination,
                source      = '',
                copy_all    = copy_all)

export_to_drive: verifying environment...
export_to_drive: no destination specified. generating default name.
export_to_drive: error: no file specified for backup.
                 select a file to back up, or set copy_all = True.


### Phase 1: Sequence acquisition and intake

#### If you are downloading multi-gene info from GenBank...

In [ ]:
# the input to voucher_matrix_to_genbank() is a CSV spreadsheet that looks like this...
#
# Voucher	      16S	     COI      H3       gene4
# CASIZ 174485  KY128712 KY128917 KY128504 AB123456
# CASIZ 179463a KY128713 KY128918 KY128505 AB789012
# etc...

# genes can be in any order, and additional unique genes can be included

# it is CRITICALLY IMPORTANT that genbank IDs be assigned correctly
#   e.g., the code can't tell if you put a COI entry in the 16S column
#   as of 0.13a, genbank_to_fastas() saves .log files that help identify
#   incorrectly classified accession numbers; it can also (slowly) run BLAST

In [ ]:
# @title voucher_matrix_to_genbank() {single-column:true}
# @markdown Input CSV filename  (see above for example formatting)
genbank_csv_filename = "" # @param {"type":"string","placeholder":"Tenellia_corrected.csv"}
voucher_matrix_to_genbank(genbank_csv_filename);

In [ ]:
# voucher_matrix_to_genbank() has 2 outputs...
#   a _clean.csv file, which has sanitized names to carry forward
#   a _BE.txt file, for upload to Batch Entrez
#      1. go to https://www.ncbi.nlm.nih.gov/sites/batchentrez
#      2. upload the _BE.txt file
#      3. download the BE result as a GenBank file (.gb), rename, and upload
#
# genbank_to_fastas() takes three inputs
#   file_prefix       is a file prefix or project name common to all input files
#   assignment_matrix is the clean CSV file produced by voucher_matrix_to_genbank()
#   genbank_input     is the .gb download from genbank

In [ ]:
# @title genbank_to_fastas() {single-column:true}
# @markdown Desired universal file prefix (i.e., project name)
file_prefix       = "" # @param {"type":"string","placeholder":"Tenellia"}
# @markdown Assignment matrix from voucher_matrix_to_genbank() (ending in _clean.csv)
assignment_matrix = "" # @param {"type":"string","placeholder":"Tenellia_corrected_clean.csv"}
# @markdown Genbank download filename (.gb)
genbank_input     = "" # @param {"type":"string","placeholder":"Tenellia_corrected.gb"}

genbank_to_fastas(file_prefix       = file_prefix,
                  assignment_matrix = assignment_matrix,
                  genbank_input     = genbank_input);

#### Optional detour: batch BLAST


In [ ]:
# BLAST_wrapper() takes three inputs
#   input_type is 'file' or 'folder'
#   input_name is the FASTA or folder name (containing FASTAs)
#   max_hits   is the maximum matches to return per query

# this runs a remote BLAST. it takes a while. go have lunch.

# BLAST_wrapper() creates a .zip archive
#   it contains one TSV file for each FASTA
#   for each sequence in that FASTA, it will display the top n hits
#   this can help identify sequences with incorrect GenBank metadata

In [ ]:
# @title BLAST_wrapper() {single-column:true}
# @markdown Input type (choose either 'folder' or 'file')
input_type = 'folder' # @param ['folder', 'file'] {"type":"string","placeholder":"Tenellia"}
# @markdown Input file or folder name
input_name = '' # @param {"type":"string","placeholder":"Tenellia_out"}
# @markdown Maximum number of hits to display (10 recommended)
max_hits   = None # @param {"type":"integer","placeholder":'10'}

BLAST_wrapper(input_type = input_type,
              input_name = input_name,
              max_hits = max_hits)

#### If you are concatenating input FASTAs for ONE GENE...

In [ ]:
# genbank_to_fasta() creates a .zip archive
#   it contains one folder
#   this contains one FASTA file per gene in the original table input
#   download it, extract it, and add your sequences to the FASTA files
#   this can be done in AliView, BioPython, Geneious, or a text editor

#   it is critical that sequences to be assigned to the same specimen have identical names
#   once sequences are added, compress this folder (.zip) and reupload
#   it is critical that sequences to be assigned to the same specimen have identical names
#   ^ i said that multiple times on purpose, because
#   it is critical that sequences to be assigned to the same specimen have identical names

# join_single_gene_FASTA() takes four arguments:
#   gene               is the gene ID, which must appear at the end of filenames (e.g., 'Tenellia_COI.fasta')
#   file_prefix        will be appended to all files
#   input_archive_name is the name of the .zip archive being uploaded
#   delim              (optional) precedes the gene ID in filenames(default '_')

In [ ]:
# join_single_gene_FASTA() takes four arguments:
#   gene               is the gene ID, which must appear at the end of filenames (e.g., 'Tenellia_COI.fasta')
#   file_prefix        will be appended to all files
#   input_archive_name is the name of the .zip archive being uploaded
#   delim              (optional) precedes the gene ID in filenames(default '_')

# @title join_single_gene_FASTA() {single-column:true}
# @markdown Name of gene (will be appended to filename)
gene               = "" # @param {"type":"string", "placeholder":"COI"}
# @markdown Desired file prefix (i.e., project name)
file_prefix        = "" # @param {"type":"string", "placeholder":"Tenellia"}
# @markdown Input archive name (.zip format)
input_archive_name = "" # @param {"type":"string", "placeholder":"my_Tenellia_fastas.zip"}

join_single_gene_FASTA(gene = gene,
                       file_prefix = file_prefix,
                       input_archive_name = input_archive_name);

#### If you are concatenating FASTAs for MULTIPLE genes...

In [ ]:
# join_multi_gene_FASTAs() takes four arguments:
#   gene_list          is a space-delimited list of gene IDs (e.g., "16S COI H3"; no quotes in Colab)
#   file_prefix        will be appended to all files
#   input_archive_name is the name of the .zip archive being uploaded
#   delim              (optional) precedes the gene ID in filenames(default '_')

In [ ]:
# if you are concatenating FASTAs for MULTIPLE genes...

# join_multi_gene_FASTAs() takes four arguments:
#   gene_list          is a
#   file_prefix        will be appended to all files
#   input_archive_name is the name of the .zip archive being uploaded
#   delim              (optional) precedes the gene ID in filenames(default '_')

# @title join_multi_gene_FASTAs() {single-column:true}
# @markdown Desired universal file prefix (i.e., project name)
file_prefix        ="" # @param {"type":"string", "placeholder":"Tenellia"}
# @markdown Space-delimited list of gene IDs (no quotes)
# THE ABOVE MARKDOWN LINE ONLY APPLIES IF YOU ARE USING THE COLAB FORMS UX.
# IF YOU ARE EDITING THE CODE DIRECTLY, INCLUDE QUOTES AS A NORMAL STRING.
gene_list          = "" # @param {"type":"string", "placeholder":"16S COI H3"}
# @markdown Input archive name (.zip format)
input_archive_name = "" # @param {"type":"string", "placeholder":"my_tenellia_fastas.zip"}

join_multi_gene_FASTAs(file_prefix = file_prefix,
                       gene_list   = gene_list.split(' '),
                       input_archive_name = input_archive_name);

#### If you are concatenating FASTAs for MULTIPLE genes, AND combining it with a GenBank download...


In [ ]:
# genbank_concatenator() takes 4 arguments:
#   file_prefix        is a file prefix (e.g., the project name)
#   assignment_matrix  is the clean CSV file produced by voucher_matrix_to_genbank()
#   genbank_input      is the .gb download from genbank
#   input_archive_name is the name of the .zip archive being uploaded

In [ ]:
# if you are concatenating FASTAs for MULTIPLE genes, AND combining it with a GenBank download...

# genbank_concatenator() takes 4 arguments:
#   file_prefix        is a file prefix (e.g., the project name)
#   assignment_matrix  is the clean CSV file produced by voucher_matrix_to_genbank()
#   genbank_input      is the .gb download from genbank
#   input_archive_name is the name of the .zip archive being uploaded

# @title genbank_concatenator() {single-column:true}
# @markdown Desired universal file prefix (i.e., project name)
file_prefix        = "" # @param {"type":"string", "placeholder":"Tenellia"}
# @markdown Assignment matrix from voucher_matrix_to_genbank() (ending in _clean.csv)
assignment_matrix  = "" # @param {"type":"string", "placeholder":"Tenellia_corrected_clean.csv"}
# @markdown Genbank download filename (.gb)
genbank_input      = "" # @param {"type":"string", "placeholder":"Tenellia_corrected.gb"}
# @markdown Input archive name (.zip format)
input_archive_name = "" # @param {"type":"string", "placeholder":"my_Tenellia_fastas.zip"}

genbank_concatenator(file_prefix        = file_prefix,
                     assignment_matrix  = assignment_matrix,
                     genbank_input      = genbank_input,
                     input_archive_name = input_archive_name);

### Phase 2: Sequence alignment with MAFFT (below; click to expand)
Note: "--adjustdirection" parameter automatically checks for reverse complements

In [ ]:
# MAFFT_wrapper() takes up to five inputs
#   input_type is 'file' or 'folder'
#   input_name is a FASTA filename, or the name of a folder containing FASTAs
#   prefix          (optional, default: input name) is the project name
#   adjustdirection (optional, default: True) MAFFT reverse-complement detection
#   concat          (optional, default: True) creates a concatenated Nexus
#                   supermatrix (.supermatrix.nex) for use in MrBayes

In [ ]:
# MAFFT_wrapper() takes two inputs
#   input_type is 'file' or 'folder'
#   input_name is a FASTA filename, or the name of a folder containing FASTAs

# @title MAFFT_wrapper() {single-column:true}

# @markdown Input type (choose 'folder' or 'file' )
input_type      = "file" # @param ["folder", "file"] {"type":"string"}

# @markdown Input file or folder name
input_name      = "" # @param {"type":"string", "placeholder":"Tenellia_concat"}

# @markdown Project name (optional; if different than input name)
prefix          = "" # @param {"type":"string"}

# @markdown Check for reverse-complemented sequences?
adjustdirection = True # @param {"type":"boolean"}

# @markdown Concatenate alignments into a single NEXUS matrix for MrBayes?
concat          = False # @param {"type":"boolean"}

MAFFT_wrapper(input_type      = input_type,
              input_name      = input_name,
              prefix          = prefix,
              concat          = concat,
              adjustdirection = adjustdirection)

MAFFT_wrapper:  initializing...

MAFFT_single:   processing 282 documents from GenBank Coryphella Fasta...

MAFFT_single:   adjustdirection = True. passing setting to MAFFT.
                282 documents from GenBank Coryphella Fasta.fna written successfully.
                MAFFT logs written to logs/282 documents from GenBank Coryphella Fasta.log.
                282 sequences, 668 columns (incl. gaps) in 282 documents from GenBank Coryphella Fasta.fna

MAFFT_single:   checking for reverse-complement (RC) tags...
                GQ292022.1 RC-tagged by MAFFT. restoring voucher ID...
                HM162694.1 RC-tagged by MAFFT. restoring voucher ID...
                HM162717.1 RC-tagged by MAFFT. restoring voucher ID...
                HM162746.1 RC-tagged by MAFFT. restoring voucher ID...
                HM162749.1 RC-tagged by MAFFT. restoring voucher ID...
                HM162758.1 RC-tagged by MAFFT. restoring voucher ID...
                HQ616748.1 RC-tagged by MAFFT. restor

In [ ]:
# MAFFT_wrapper() creates at least one output:
#   if input_type = 'file', this will be a single aligned FASTA (.afa)
#   if input_type = 'folder', it will be a .ZIP archive containing...
#   - a  "logs" folder, containing MAFFT logs (.log)
#   - an "afa"  folder, containing aligned FASTAs (.afa)
#   - a  "nex"  folder, containing single NEXUS alignments (.nex)
#     if "concat" = True, then there will be a combined file (.supermatrix.nex)

# CAUTION: VISUALLY CHECK YOUR ALIGNMENTS! (e.g., in AliView, Mesquite, etc)
#   the code cannot tell if a sequence is mis-assigned
#     (e.g., if a COI sequence is mixed in with H3 sequences)
#   these errors are typically obvious in alignment viewers and will appear as
#     outliers: one or two very long/short sequences that cause extreme gapping.

#   check the sequences and trim as you see fit
#   BLAST any suspicious-looking sequences to check for incorrect genbank accessions
#   reverse-complementing sequences may be necessary to get a proper alignment
#     (as of v0.13+, this can be handled atomatically by MAFFT)

### Phase 3: Building and cleaning phylogenetic trees (click to expand)

#### IQ-TREE: Maximum-Likelihood (ML) phylogenetic analysis

In [ ]:
# iqtree_wrapper() takes at least two inputs...
#   prefix     is your project name
#   input      is a folder of aligned FASTAs
#   bootstraps (optional) is the number of bootstraps (default: 1000)
#   model      (optional) is the model to use (default: MFP)

In [ ]:
# @title iqtree_wrapper() {single-column:true}
# @markdown Desired output file prefix (i.e., project name)
prefix = '' # @param {"type":"string", "placeholder":"Tenellia"}
# @markdown Input (single alignment or folder containing alignment files)
input = '' # @param {"type":"string", "placeholder":"Tenellia_concat_aligned/afa"}
# @markdown Number of bootstraps (Optional; Default: 1000)
bootstraps = None # @param {"type":"integer", "placeholder":'1000'}
# @markdown Evolutionary model (Optional; Default: MFP)
model = "" # @param {"type":"string", "placeholder":'MFP'}
# @markdown Number of threads (Optional; Default: AUTO)
nthreads = "" # @param {"type":"string", "placeholder":'AUTO'}
# @markdown Overwrite a previously completed run (Optional; Default: False)?
redo      = False # @param {"type":"boolean", "placeholder":"False"}


iqtree_wrapper(prefix = prefix,
               input  = input,
               bootstraps = bootstraps or 1000,
               model = model or 'MFP',
               nthreads = nthreads or 'AUTO',
               redo = redo)

  attempting to run IQTree with the following parameters:
  /content/bin/iqtree-3.1.3-Linux/bin/iqtree3 -s /content/afa/282 documents from GenBank Coryphella Fasta.fna -pre Megan_iqtree/Megan -m MFP -bb 1000 -nt AUTO
IQ-TREE version 3.1.3 for Linux x86 64-bit built Jun 19 2026
Developed by Bui Quang Minh, Thomas Wong, Nhan Ly-Trong, Huaiyan Ren
Contributed by Lam-Tung Nguyen, Dominik Schrempf, Chris Bielow,
Olga Chernomor, Michael Woodhams, Diep Thi Hoang, Heiko Schmidt

Host:    59230b4fddb3 (AVX2, FMA3, 12 GB RAM)
Command: /content/bin/iqtree-3.1.3-Linux/bin/iqtree3_intel -s /content/afa/282 documents from GenBank Coryphella Fasta.fna -pre Megan_iqtree/Megan -m MFP -bb 1000 -nt AUTO
Seed:    173403 (Using SPRNG - Scalable Parallel Random Number Generator)
Time:    Mon Aug 24 22:26:54 2026
Kernel:  AVX+FMA - auto-detect threads (2 CPU cores detected)

Reading alignment file /content/afa/282 documents from GenBank Coryphella Fasta.fna ... Fasta format detected
Reading fasta file: done 

#### MrBayes: Bayesian Inference (BI) phylogenetic analysis

In [ ]:
# MrBayes_wrapper() takes several inputs:
#   prefix      is the desired output file prefix (i.e., project name)
#   input_nex   is the path to the input Nexus alignment file (.nex)
#   runs        (Optional) is the number of MCMC analyses to run; default 2)
#   ncahins     (Optional) is the number of MCMC chains to use; default: 4)
#   burnin      (Optional) is the burn-in time; default 0.25
#               this is an integer (n generations) OR a decimal (0.25)
#   generations (Optional) is the number of generations; default: 1000000)
#   samplefreq  (Optional) is the MCMC sample frequency; default 1000)
#   partitioned (Optional) toggles partitioned analysis on/off; default: False

In [ ]:
# @title MrBayes_wrapper() {single-column:true}
# @markdown Desired output file prefix (i.e., project name)
prefix    = "" # @param {"type":"string", "placeholder":"Tenellia"}
# @markdown Path to input Nexus alignment file (.nex)
input_nex = "" # @param {"type":"string", "placeholder":"./Tenellia_concat_aligned/nex/Tenellia_concat_aligned.supermatrix.nex"}
# @markdown Path to predefined Bayes block file (If you don't have one, leave blank and set the next parameters)
input_block = "" # @param {"type":"string", "placeholder":"./Tenellia_concat_aligned/nex/Tenellia_concat_aligned.bayesblock.nex"}

# @markdown Number of MCMC analyses to run (Optional; default 2)
runs = None    # @param {"type":"integer", "placeholder":"2"}
# @markdown Number of MCMC chains to use (Default: 4)
nchains = None # @param {"type":"integer", "placeholder":"4"}
# @markdown Burn-in time (Optional; number of generations OR a decimal value; Default 0.25)
burnin = "" # @param {"type":"string", "placeholder":"0.25"}
# @markdown Number of generations (Optional; Default: 1000000)
generations = None # @param {"type":"integer", "placeholder":"1000000"}
# @markdown MCMC sample frequency (Optional; default 1000)
samplefreq = None # @param {"type":"integer", "placeholder":"1000"}
# @markdown Is this a partitioned analysis? Default: False
partitioned = True # @param {"type":"boolean", "placeholder":"False"}
# @markdown Attempt to resume a previously aborted run from checkpoint file? Default: False
resume      = True # @param {"type":"boolean", "placeholder":"False"}

MrBayes_wrapper(prefix    = prefix,
                input_nex = input_nex,
                input_block = input_block,
                runs = runs or 2, nchains = nchains or 4,
                burnin = burnin,
                generations = generations or 1000000,
                samplefreq = samplefreq or 1000,
                partitioned = partitioned,
                resume = resume)

#### Optional feature: Rename leaves on your output tree

In [ ]:
# leaf_renamer takes several inputs:
#   tree_filepath is the name of a Newick tree file
#   sample_table  is the name of a  two-column .CSV file
#   old_names     is the name of the column containing the names as they CURRENTLY appear
#   new_names     is the name of the column containing the names as they SHOULD appear
#   sample_suffix    (optional) is any text string to be removed from the end of all samples
#   trim_parenthesis (optional) if True, will trim gently:   A (B) C (D) → A_C
#   greedy_trimming  (optional) if True, will trim greedily: A (B) C (D) → A

In [ ]:
# @title leaf_renamer() {single-column:true}
# @markdown Newick tree filename
tree_filepath = "" # @param {"type":"string", "placeholder":"Tenellia_concat.contree"}
# @markdown Filename of two-column CSV table with old and new filenames
sample_table = "" # @param {"type":"string", "placeholder":"Tenellia_names.csv"}
# @markdown Name of column containing ORIGINAL sample names (e.g., from voucher_matrix_to_genbank())
old_names = "" # @param {"type":"string", "placeholder":"Voucher_old"}
# @markdown Name of column containing DESIRED sample names
new_names = "" # @param {"type":"string", "placeholder":"Voucher_new"}
# @markdown Suffix (any undesired text currently appended to all samples)
sample_suffix = "" # @param {"type":"string", "placeholder":"_Tenellia"}
# @markdown Trim parenthesis (e.g., citation information?)
trim_parenthesis = False # @param {"type":"boolean"}
# @markdown Trim greedily?
greedy_trimming = False # @param {"type":"boolean"}

leaf_renamer(tree_filepath    = tree_filepath,
             sample_table     = sample_table,
             old_names        = old_names,
             new_names        = new_names,
             sample_suffix    = sample_suffix,
             trim_parenthesis = trim_parenthesis,
             greedy_trimming  = greedy_trimming)

leaf_renamer:           reading Megan.contree...

leaf_renamer:           calling sanitize_fasta_headers()...
sanitize_fasta_headers: removing MAFFT/IQ-TREE-breaking characters...

leaf_renamer:           removing Newick-breaking characters...

leaf_renamer:           checking for duplicate vouchers...

leaf_renamer:           checking for duplicate vouchers...

leaf_renamer:           saving output file to:
                        /content/Megan_iqtree/renamed_Megan.contree
